# Preprocess And Align

This notebook converts raw uploads into standardized processed JPEGs, aligns each location group, filters duplicate conditions down to one representative, and writes a final `manifest.csv` for diffusion fine-tuning.

## Runtime Setup

Mount Google Drive when running in Colab, install the image-processing stack, and install the GitHub LightGlue package used for feature matching during alignment.

In [ ]:
# Colab/runtime setup.
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

!pip install -q uv
!uv pip install --system git+https://github.com/cvg/LightGlue.git pillow pillow-heif pandas opencv-python-headless torch torchvision matplotlib tqdm


## Imports And Helpers

Load common libraries and define small utility functions used across every stage: filename cleaning, stable caption generation, deterministic train/validation splitting, directory resets, and manifest writing.

In [ ]:
import gc
import json
import os
import re
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps, UnidentifiedImageError
from pillow_heif import register_heif_opener
from lightglue import LightGlue, SuperPoint
from lightglue.utils import load_image
from tqdm.auto import tqdm

register_heif_opener()
plt.rcParams['figure.figsize'] = (14, 7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

MANIFEST_COLUMNS = [
    'file_name',
    'location',
    'time_of_day',
    'weather',
    'is_synthetic',
    'caption',
    'source_file',
    'original_file_name',
    'anchor_file',
    'crop_w',
    'crop_h',
    'crop_area_ratio',
    'matches',
    'inliers',
    'inlier_ratio',
    'representative_score',
    'split',
    'status',
    'drop_reason',
]

VALID_IMAGE_SUFFIXES = {'.heic', '.heif', '.jpg', '.jpeg', '.png'}
FILENAME_PATTERN = re.compile(
    r'^(?P<location>[A-Za-z0-9]+)_(?P<time>[A-Za-z0-9]+)_(?P<weather>[A-Za-z0-9]+)(?P<rest>.*?)(?P<synthetic>_ai)?$',
    re.IGNORECASE,
)
TIME_ALIASES = {
    'day': 'daytime',
    'daytime': 'daytime',
    'morning': 'daytime',
    'afternoon': 'daytime',
    'noon': 'daytime',
    'sunset': 'sunset',
    'dusk': 'sunset',
    'evening': 'sunset',
    'night': 'night',
    'nighttime': 'night',
}
WEATHER_ALIASES = {
    'clear': 'clear',
    'sunny': 'clear',
    'cloudy': 'cloudy',
    'overcast': 'cloudy',
    'rain': 'rainy',
    'rainy': 'rainy',
    'wet': 'rainy',
    'snow': 'snowy',
    'snowy': 'snowy',
}


def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def clean_stem(value):
    stem = Path(str(value)).stem.lower().strip()
    stem = re.sub(r'\s+', '_', stem)
    stem = re.sub(r'[^a-z0-9_()\-]+', '_', stem)
    return stem.strip('_') or 'image'


def clean_label(value):
    return re.sub(r'[^a-z0-9]+', ' ', str(value).lower()).strip()


def caption_from_labels(location, time_of_day, weather):
    loc = clean_label(location)
    tod = clean_label(time_of_day)
    wx = clean_label(weather)
    return f'A photo of {loc} at Penn during {tod} {wx} weather.'


def split_for_location(location):
    # Deterministic location-level split so variants of the same scene stay together.
    return 'val' if sum(ord(ch) for ch in str(location).lower()) % 5 == 0 else 'train'


def replace_dir(path):
    if os.path.isdir(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def file_exists_and_nonempty(path):
    return os.path.exists(path) and os.path.getsize(path) > 0




def manifest_record_from_row(row, status, drop_reason='', file_name=None, stats=None, crop_w='', crop_h='', crop_area_ratio='', representative_score='', anchor_file=''):
    stats = stats or {}
    return {
        'file_name': file_name if file_name is not None else row.get('file_name', row.get('source_file', '')),
        'location': row.get('location', ''),
        'time_of_day': row.get('time_of_day', ''),
        'weather': row.get('weather', ''),
        'is_synthetic': row.get('is_synthetic', ''),
        'caption': row.get('caption', caption_from_labels(row.get('location', ''), row.get('time_of_day', ''), row.get('weather', ''))),
        'source_file': row.get('source_file', row.get('file_name', '')),
        'original_file_name': row.get('original_file_name', ''),
        'anchor_file': anchor_file if anchor_file != '' else row.get('anchor_file', ''),
        'crop_w': crop_w if crop_w != '' else row.get('crop_w', ''),
        'crop_h': crop_h if crop_h != '' else row.get('crop_h', ''),
        'crop_area_ratio': crop_area_ratio if crop_area_ratio != '' else stats.get('crop_area_ratio', row.get('crop_area_ratio', '')),
        'matches': stats.get('matches', row.get('matches', '')),
        'inliers': stats.get('inliers', row.get('inliers', '')),
        'inlier_ratio': round(float(stats.get('inlier_ratio', row.get('inlier_ratio', 0) or 0)), 4) if stats or row.get('inlier_ratio', '') != '' else '',
        'representative_score': representative_score if representative_score != '' else row.get('representative_score', ''),
        'split': row.get('split', split_for_location(row.get('location', ''))),
        'status': status,
        'drop_reason': drop_reason,
    }

def write_manifest(df):
    out = df.copy()
    for col in MANIFEST_COLUMNS:
        if col not in out.columns:
            out[col] = ''
    out = out[MANIFEST_COLUMNS]
    out.to_csv(MANIFEST_CSV, index=False)
    return out


def summarize_manifest(df, title, path_base=None):
    print(f'\n{title}')
    print('-' * len(title))
    print('rows:', len(df))
    if 'split' in df.columns and len(df):
        print('split:', df['split'].value_counts().to_dict())
    if 'status' in df.columns and len(df):
        print('status:', df['status'].value_counts().to_dict())
    if {'time_of_day', 'weather'}.issubset(df.columns) and len(df):
        print('time/weather:')
        display(df.groupby(['time_of_day', 'weather']).size().reset_index(name='count').head(12))
    if 'location' in df.columns and len(df):
        print('top locations:')
        display(df['location'].value_counts().head(10).rename_axis('location').reset_index(name='count'))
    if path_base is not None and 'file_name' in df.columns and len(df):
        existing = sum(os.path.exists(os.path.join(path_base, str(p))) for p in df['file_name'])
        print('existing files:', existing, '/', len(df))


def show_image_sample(df, path_base, path_col='file_name', title='Sample images', n=4):
    if df is None or df.empty or path_col not in df.columns:
        print(f'{title}: no rows to preview.')
        return
    rows = df.head(n)
    fig, axes = plt.subplots(1, len(rows), figsize=(4 * len(rows), 4))
    if len(rows) == 1:
        axes = [axes]
    shown = 0
    for ax, (_, row) in zip(axes, rows.iterrows()):
        path = os.path.join(path_base, str(row[path_col]))
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            ax.set_title('missing')
            ax.axis('off')
            continue
        ax.imshow(bgr_to_rgb(img))
        label = str(row.get('location', ''))
        if 'time_of_day' in row and 'weather' in row:
            label = f"{label}\n{row['time_of_day']}/{row['weather']}"
        ax.set_title(label)
        ax.axis('off')
        shown += 1
    fig.suptitle(title)
    plt.show()
    if shown == 0:
        print(f'{title}: no readable files in sample.')


## Configuration

Set project paths, rebuild flags, model thresholds, crop-loss rules, duplicate-filter scoring knobs, and the fixed final training resolution. By default every stage rebuilds so new uploads are always included.

In [ ]:
# Main settings.
BASE_CANDIDATES = [
    '/content/drive/MyDrive/CIS_5190_group_project',
    '/content/drive/My Drive/CIS_5190_group_project',
    os.path.abspath('../data'),
]
BASE = next((p for p in BASE_CANDIDATES if os.path.exists(p)), BASE_CANDIDATES[0])

RAW_DIR = os.path.join(BASE, 'Images')
PROCESSED_DIR = os.path.join(BASE, 'processedImages') if 'content' in BASE else os.path.join(BASE, 'processed')
ALIGNED_DIR = os.path.join(BASE, 'aligned')
FILTERED_DIR = os.path.join(BASE, 'filtered_aligned')
MANIFEST_CSV = os.path.join(BASE, 'manifest.csv')
HF_DATASET_DIR = os.path.join(BASE, 'hf_dataset')

# Defaults are True so newly uploaded/replaced images are always swept into the pipeline.
REDO_PREPROCESS = True
REDO_ALIGNMENT = True
REDO_FILTERED = True
REDO_HF_DATASET = True

# Use a list like ['34th', 'agh3rd'] while debugging, or None for every location.
LOCATION_FILTER = None

# Fixed final training resolution.
OUTPUT_SIZE = 512
JPEG_QUALITY = 95

# Alignment settings.
RESIZE = 1024
MAX_KEYPOINTS = 2048
RANSAC_REPROJ_THRESHOLD = 5.0
MIN_MATCHES = 12
MIN_INLIERS = 8
MIN_INLIER_RATIO = 0.0
MIN_SHARED_CROP_AREA_RATIO = 0.50

# Duplicate filtering settings. Used after alignment to keep one image per location/time/weather.
ECC_SCORE_SIZE = 768
ECC_MAX_ITERS = 80
ECC_EPS = 1e-5
ECC_MAX_WORSE_FACTOR = 1.05
ECC_MAX_CORNER_DRIFT_FRAC = 0.20
ECC_MIN_SCALE = 0.70
ECC_MAX_SCALE = 1.30

print('Base:', BASE)
print('Raw images:', RAW_DIR)
print('Processed images:', PROCESSED_DIR)
print('Aligned images:', ALIGNED_DIR)
print('Filtered aligned images:', FILTERED_DIR)
print('Manifest CSV:', MANIFEST_CSV)
print('HF dataset:', HF_DATASET_DIR)


## Preprocess Raw Images

This stage converts raw images into orientation-corrected JPEGs under `processedImages/` and initializes `manifest.csv`. If `REDO_PREPROCESS=True`, existing processed files and the manifest are rebuilt from scratch.

### Preprocess Implementation

Parse raw filenames into labels, convert HEIC/JPEG/PNG uploads into processed JPEGs, initialize stable captions, and write the first version of `manifest.csv`. The manifest starts with processed image paths and is rewritten by later stages.

In [ ]:
def parse_filename(fname):
    stem = Path(fname).stem.strip()
    stem = re.sub(r'\s*\((\d+)\)\s*$', r'_\1', stem)
    match = FILENAME_PATTERN.match(stem)
    if not match:
        return None
    location = match.group('location').lower()
    raw_tod = match.group('time').lower()
    raw_weather = match.group('weather').lower()
    tod = TIME_ALIASES.get(raw_tod)
    weather = WEATHER_ALIASES.get(raw_weather)
    if tod is None or weather is None:
        return None
    rest_parts = [
        re.sub(r'[^a-z0-9]', '', part.lower())
        for part in re.split(r'[_\s]+', match.group('rest').strip('_ '))
    ]
    is_synthetic = bool(match.group('synthetic')) or 'ai' in {part for part in rest_parts if part}
    return location, tod, weather, is_synthetic


def next_processed_rel_path(tod, weather, location, is_synthetic, used_paths):
    condition_dir = f'{tod}_{weather}'
    location_dir = clean_stem(location)
    base_stem = f'{clean_stem(location)}_{clean_stem(tod)}_{clean_stem(weather)}'

    def candidate_name(index):
        if is_synthetic:
            return f'{base_stem}_{index}_ai.jpg'
        if index is None:
            return f'{base_stem}.jpg'
        return f'{base_stem}_{index}.jpg'

    indices = [None] if not is_synthetic else []
    indices.extend(range(1, 10000))
    for index in indices:
        rel_path = os.path.join(location_dir, condition_dir, candidate_name(index)).lower()
        if rel_path not in used_paths and not os.path.exists(os.path.join(PROCESSED_DIR, rel_path)):
            used_paths.add(rel_path)
            return rel_path
    raise RuntimeError(f'Could not allocate processed filename for {location}/{tod}/{weather}')


def preprocess_complete():
    if not file_exists_and_nonempty(MANIFEST_CSV):
        return False
    try:
        df = pd.read_csv(MANIFEST_CSV)
    except Exception:
        return False
    required = {'file_name', 'source_file', 'original_file_name', 'location', 'time_of_day', 'weather', 'is_synthetic', 'caption', 'split'}
    if not required.issubset(df.columns) or df.empty:
        return False
    old_location_index_paths = [
        p
        for p, loc in zip(df['source_file'].astype(str), df['location'].astype(str))
        if Path(p).parts and Path(p).parts[0] != clean_stem(loc)
    ]
    if old_location_index_paths:
        print('Processed manifest uses old location-index paths:', old_location_index_paths[:5])
        return False
    missing = [p for p in df['source_file'].astype(str) if not os.path.exists(os.path.join(PROCESSED_DIR, p))]
    if missing:
        print('Processed files missing from existing manifest:', missing[:5])
        return False
    return True


def run_preprocess():
    assert os.path.isdir(RAW_DIR), f'Missing raw image directory: {RAW_DIR}'
    if REDO_PREPROCESS:
        print('REDO_PREPROCESS=True; deleting previous processed images and manifest.')
        replace_dir(PROCESSED_DIR)
        if os.path.exists(MANIFEST_CSV):
            os.remove(MANIFEST_CSV)
    elif preprocess_complete():
        print('Preprocess already complete; skipping.')
        return pd.read_csv(MANIFEST_CSV)
    else:
        os.makedirs(PROCESSED_DIR, exist_ok=True)

    parsed_files = []
    skipped_non_images = []
    badly_named = []
    skipped_unreadable = []

    for fname in sorted(os.listdir(RAW_DIR)):
        suffix = os.path.splitext(fname)[1].lower()
        if ':zone.identifier' in fname.lower() or suffix not in VALID_IMAGE_SUFFIXES:
            skipped_non_images.append(fname)
            continue
        parsed = parse_filename(fname)
        if parsed is None:
            badly_named.append(fname)
            continue
        parsed_files.append((fname, *parsed))

    records = []
    used_rel_paths = set()

    for fname, location, tod, weather, is_synthetic in tqdm(parsed_files, desc='Preprocess images'):
        in_path = os.path.join(RAW_DIR, fname)
        rel_path = next_processed_rel_path(tod, weather, location, is_synthetic, used_rel_paths)
        out_path = os.path.join(PROCESSED_DIR, rel_path)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        try:
            img = ImageOps.exif_transpose(Image.open(in_path)).convert('RGB')
            img.save(out_path, 'JPEG', quality=JPEG_QUALITY)
        except (UnidentifiedImageError, OSError) as exc:
            skipped_unreadable.append((fname, str(exc)))
            continue

        records.append({
            'file_name': rel_path,
            'location': location,
            'time_of_day': tod,
            'weather': weather,
            'is_synthetic': is_synthetic,
            'caption': caption_from_labels(location, tod, weather),
            'source_file': rel_path,
            'original_file_name': fname.lower(),
            'anchor_file': '',
            'crop_w': '',
            'crop_h': '',
            'crop_area_ratio': '',
            'matches': '',
            'inliers': '',
            'inlier_ratio': '',
            'representative_score': '',
            'split': split_for_location(location),
            'status': 'processed',
            'drop_reason': '',
        })

    df = pd.DataFrame(records).sort_values(['location', 'time_of_day', 'weather', 'source_file'])
    df = write_manifest(df)
    print(f'Saved {len(df)} rows to {MANIFEST_CSV}')
    if badly_named:
        print(f'Badly named files ({len(badly_named)}):', badly_named[:20])
    if skipped_non_images:
        print(f'Skipped non-image sidecar/files ({len(skipped_non_images)}):', skipped_non_images[:20])
    if skipped_unreadable:
        print(f'Skipped unreadable image files ({len(skipped_unreadable)}):', skipped_unreadable[:10])
    return df


manifest_df = run_preprocess()
display(manifest_df.head())
summarize_manifest(manifest_df, 'Preprocess summary', path_base=PROCESSED_DIR)
show_image_sample(manifest_df, PROCESSED_DIR, path_col='source_file', title='Processed image sample')


## Align Processed Images

This stage aligns every image in a location group to one anchor. It removes black/empty warp areas by applying one uniform crop per group. If including a warped image would force the shared crop below `MIN_SHARED_CROP_AREA_RATIO`, that image is dropped and the crop is recomputed for the remaining group.

### Matching And Homography Helpers

Load processed images, run SuperPoint + LightGlue matching, estimate target-to-anchor homographies with MAGSAC, and compute shared valid crops. These helpers do not save files directly; saving happens in the group alignment stage below.

In [ ]:
DAYLIKE = {'daytime', 'day', 'morning'}
alignment_failures = []
crop_drops = []


def alignment_complete():
    if not file_exists_and_nonempty(MANIFEST_CSV):
        return False
    try:
        df = pd.read_csv(MANIFEST_CSV)
    except Exception:
        return False
    if df.empty or 'file_name' not in df.columns:
        return False
    if 'status' not in df.columns:
        return False
    expected_statuses = {'aligned', 'alignment_failed', 'crop_dropped'}
    statuses = set(df['status'].dropna().astype(str))
    if not statuses or not statuses.issubset(expected_statuses):
        return False
    aligned_rows = df[df['file_name'].astype(str).str.startswith('aligned/')]
    if aligned_rows.empty:
        return False
    missing = [p for p in aligned_rows['file_name'].astype(str) if not os.path.exists(os.path.join(BASE, p))]
    if missing:
        print('Aligned files missing from existing manifest:', missing[:5])
        return False
    return True


def pick_anchor(group):
    tod = group['time_of_day'].astype(str).str.lower().str.strip()
    weather = group['weather'].astype(str).str.lower().str.strip()
    candidates = group[tod.isin(DAYLIKE) & (weather == 'clear')]
    if candidates.empty:
        candidates = group[tod.isin(DAYLIKE)]
    if candidates.empty:
        candidates = group
    return candidates.iloc[0]


def load_bgr_checked(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img


def lightglue_match(anchor_path, target_path):
    image0_raw = load_image(anchor_path, resize=RESIZE)
    image1_raw = load_image(target_path, resize=RESIZE)
    image0 = image0_raw.mean(dim=0, keepdim=True).unsqueeze(0).to(device)
    image1 = image1_raw.mean(dim=0, keepdim=True).unsqueeze(0).to(device)

    with torch.inference_mode():
        feats0 = extractor({'image': image0})
        feats1 = extractor({'image': image1})
        matches01 = matcher({'image0': feats0, 'image1': feats1})

    kpts0 = feats0['keypoints'][0].detach().cpu().numpy()
    kpts1 = feats1['keypoints'][0].detach().cpu().numpy()
    matches = matches01['matches'][0].detach().cpu().numpy()
    if len(matches) == 0:
        return None, {'matches': 0, 'inliers': 0, 'inlier_ratio': 0.0, 'reason': 'no_matches'}
    return (image0_raw, image1_raw, kpts0[matches[:, 0]], kpts1[matches[:, 1]], len(matches)), None


def estimate_target_to_anchor(anchor_bgr, target_bgr, anchor_path, target_path):
    match_result, error = lightglue_match(anchor_path, target_path)
    if error is not None:
        return None, error

    image0_raw, image1_raw, mkpts0, mkpts1, n_matches = match_result
    if n_matches < MIN_MATCHES:
        return None, {'matches': n_matches, 'inliers': 0, 'inlier_ratio': 0.0, 'reason': 'not_enough_matches'}

    h0, w0 = anchor_bgr.shape[:2]
    h1, w1 = target_bgr.shape[:2]
    scale0 = np.array([w0 / image0_raw.shape[2], h0 / image0_raw.shape[1]])
    scale1 = np.array([w1 / image1_raw.shape[2], h1 / image1_raw.shape[1]])
    mkpts0_orig = mkpts0 * scale0
    mkpts1_orig = mkpts1 * scale1

    H, inlier_mask = cv2.findHomography(mkpts1_orig, mkpts0_orig, cv2.USAC_MAGSAC, RANSAC_REPROJ_THRESHOLD)
    if H is None or inlier_mask is None:
        return None, {'matches': n_matches, 'inliers': 0, 'inlier_ratio': 0.0, 'reason': 'homography_failed'}

    inliers = int(inlier_mask.ravel().sum())
    inlier_ratio = inliers / max(n_matches, 1)
    if inliers < MIN_INLIERS:
        return None, {'matches': n_matches, 'inliers': inliers, 'inlier_ratio': inlier_ratio, 'reason': 'too_few_inliers'}
    if MIN_INLIER_RATIO > 0 and inlier_ratio < MIN_INLIER_RATIO:
        return None, {'matches': n_matches, 'inliers': inliers, 'inlier_ratio': inlier_ratio, 'reason': 'too_low_inlier_ratio'}

    return H, {'matches': n_matches, 'inliers': inliers, 'inlier_ratio': inlier_ratio, 'reason': 'ok'}


def resize_output(img):
    return cv2.resize(img, (OUTPUT_SIZE, OUTPUT_SIZE), interpolation=cv2.INTER_AREA)


def unique_output_name(location_dir, stem):
    base = f'{stem}_aligned.jpg'
    candidate = base
    counter = 2
    while os.path.exists(os.path.join(location_dir, candidate)):
        candidate = f'{stem}_aligned_{counter}.jpg'
        counter += 1
    return candidate


def shared_crop_for_items(items, anchor_area):
    shared_mask = items[0]['mask']
    for item in items[1:]:
        shared_mask = cv2.bitwise_and(shared_mask, item['mask'])
    coords = cv2.findNonZero(shared_mask)
    if coords is None:
        return None, 0.0
    x, y, crop_w, crop_h = cv2.boundingRect(coords)
    return (x, y, crop_w, crop_h), (crop_w * crop_h) / max(1, anchor_area)


def prune_to_valid_shared_crop(items, anchor_area, anchor_source):
    dropped = []
    while True:
        crop, area_ratio = shared_crop_for_items(items, anchor_area)
        if crop is not None and area_ratio >= MIN_SHARED_CROP_AREA_RATIO:
            return items, dropped, crop, area_ratio
        if len(items) <= 1:
            return items, dropped, crop, area_ratio

        best_idx = None
        best_area = -1.0
        for idx in range(1, len(items)):
            trial = items[:idx] + items[idx + 1:]
            _, trial_area = shared_crop_for_items(trial, anchor_area)
            if trial_area > best_area:
                best_idx = idx
                best_area = trial_area
        dropped_item = items.pop(best_idx)
        drop_stats = {
            **dropped_item['stats'],
            'crop_area_ratio': round(float(area_ratio or 0), 4),
            'best_without_drop_area_ratio': round(float(best_area or 0), 4),
        }
        dropped.append(dropped_item)
        crop_drops.append(manifest_record_from_row(
            dropped_item['row'],
            'crop_dropped',
            'excessive_crop_loss',
            stats=drop_stats,
            anchor_file=anchor_source,
        ))
        print(f"  [DROP] {dropped_item['row']['source_file']} caused excessive crop loss")


### Group Alignment

For each location, choose one anchor, warp all successfully matched images into the anchor coordinate system, remove black/empty warp regions with one shared crop, and drop images that would force excessive crop loss.

In [ ]:
def align_location_group(location, group):
    group = group.copy().reset_index(drop=True)
    anchor_row = pick_anchor(group)
    anchor_source = anchor_row['source_file']
    anchor_path = os.path.join(PROCESSED_DIR, anchor_source)

    try:
        anchor_bgr = load_bgr_checked(anchor_path)
    except FileNotFoundError:
        print(f'[SKIP] {location}: missing anchor {anchor_path}')
        return []

    h0, w0 = anchor_bgr.shape[:2]
    anchor_area = h0 * w0
    aligned_items = [{
        'row': anchor_row,
        'image': anchor_bgr,
        'mask': np.ones((h0, w0), dtype=np.uint8) * 255,
        'stats': {'matches': 0, 'inliers': 0, 'inlier_ratio': 1.0, 'reason': 'anchor'},
        'is_anchor': True,
    }]

    print(f'\nProcessing {location}: {len(group)} images | anchor={anchor_source}')
    for _, row in group.iterrows():
        if row['source_file'] == anchor_source:
            continue

        target_source = row['source_file']
        target_path = os.path.join(PROCESSED_DIR, target_source)
        try:
            target_bgr = load_bgr_checked(target_path)
        except FileNotFoundError:
            alignment_failures.append(manifest_record_from_row(row, 'alignment_failed', 'missing_file', stats={'matches': 0, 'inliers': 0, 'inlier_ratio': 0.0}, anchor_file=anchor_source))
            print(f'  [MISS] {target_source}')
            continue

        H, stats = estimate_target_to_anchor(anchor_bgr, target_bgr, anchor_path, target_path)
        if H is None:
            alignment_failures.append(manifest_record_from_row(row, 'alignment_failed', stats['reason'], stats=stats, anchor_file=anchor_source))
            print(f"  [FAIL] {target_source} matches={stats['matches']} inliers={stats['inliers']} ratio={stats['inlier_ratio']:.2f} reason={stats['reason']}")
            continue

        aligned = cv2.warpPerspective(target_bgr, H, (w0, h0))
        source_mask = np.ones(target_bgr.shape[:2], dtype=np.uint8) * 255
        warped_mask = cv2.warpPerspective(source_mask, H, (w0, h0))
        aligned_items.append({
            'row': row,
            'image': aligned,
            'mask': warped_mask,
            'stats': stats,
            'is_anchor': False,
        })
        print(f"  [OK] {target_source} matches={stats['matches']} inliers={stats['inliers']} ratio={stats['inlier_ratio']:.2f}")

    aligned_items, dropped, crop, area_ratio = prune_to_valid_shared_crop(aligned_items, anchor_area, anchor_source)
    if crop is None:
        print(f'  [SKIP] {location}: no shared crop after pruning')
        return []

    x, y, crop_w, crop_h = crop
    location_dir = os.path.join(ALIGNED_DIR, clean_stem(location))
    os.makedirs(location_dir, exist_ok=True)

    records = []
    for item in aligned_items:
        row = item['row']
        cropped = resize_output(item['image'][y:y + crop_h, x:x + crop_w])
        out_name = unique_output_name(location_dir, clean_stem(row.get('original_file_name', row['source_file'])))
        out_path = os.path.join(location_dir, out_name)
        cv2.imwrite(out_path, cropped, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        rel_file = os.path.relpath(out_path, BASE)

        stats = item['stats']
        records.append(manifest_record_from_row(
            row,
            'aligned',
            file_name=rel_file,
            stats=stats,
            crop_w=crop_w,
            crop_h=crop_h,
            crop_area_ratio=round(float(area_ratio or 0), 4),
            anchor_file=anchor_source,
        ))

    print(f'  [SAVE] {len(records)} aligned images | crop=({x}, {y}, {crop_w}, {crop_h}) area={area_ratio:.2f} dropped={len(dropped)}')
    return records


def run_alignment(manifest_df):
    global alignment_failures, crop_drops
    alignment_failures = []
    crop_drops = []
    if REDO_ALIGNMENT:
        print('REDO_ALIGNMENT=True; deleting previous aligned outputs.')
        replace_dir(ALIGNED_DIR)
    elif alignment_complete():
        print('Alignment already complete; skipping.')
        return pd.read_csv(MANIFEST_CSV)
    else:
        os.makedirs(ALIGNED_DIR, exist_ok=True)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    global extractor, matcher
    extractor = SuperPoint(max_num_keypoints=MAX_KEYPOINTS).eval().to(device)
    matcher = LightGlue(features='superpoint').eval().to(device)

    df = manifest_df.copy()
    required_cols = {'source_file', 'location', 'time_of_day', 'weather'}
    missing_cols = required_cols - set(df.columns)
    assert not missing_cols, f'Missing columns: {missing_cols}'

    if LOCATION_FILTER is not None:
        wanted = {str(x).lower().strip() for x in LOCATION_FILTER}
        df = df[df['location'].astype(str).str.lower().str.strip().isin(wanted)].copy()

    all_records = []
    for location, group in tqdm(list(df.groupby('location', sort=True)), desc='Align locations'):
        all_records.extend(align_location_group(location, group))

    all_records.extend(alignment_failures)
    all_records.extend(crop_drops)
    out_df = write_manifest(pd.DataFrame(all_records))
    print(f'\nAlignment stage wrote {len(out_df)} rows to {MANIFEST_CSV}')
    return out_df


aligned_df = run_alignment(manifest_df)
display(aligned_df.head())
summarize_manifest(aligned_df, 'Alignment summary', path_base=BASE)
print('failed alignments:', len(alignment_failures))
print('dropped for crop loss:', len(crop_drops))
if alignment_failures:
    display(pd.DataFrame(alignment_failures)[['location', 'source_file', 'anchor_file', 'drop_reason', 'matches', 'inliers', 'inlier_ratio']].head(10))
if crop_drops:
    display(pd.DataFrame(crop_drops)[['location', 'source_file', 'anchor_file', 'drop_reason', 'matches', 'inliers', 'inlier_ratio', 'crop_area_ratio']].head(10))
show_image_sample(aligned_df, BASE, title='Aligned image sample')


### Duplicate Representative Selection

After full alignment, reduce duplicate copies of the same `location + time_of_day + weather` condition to one representative. The extra ECC homography is used only for scoring consistency among duplicates; it does not rewrite the final image geometry.

In [ ]:
def filtered_complete():
    if not file_exists_and_nonempty(MANIFEST_CSV):
        return False
    try:
        df = pd.read_csv(MANIFEST_CSV)
    except Exception:
        return False
    if df.empty or 'file_name' not in df.columns:
        return False
    filtered_rows = df[df['file_name'].astype(str).str.startswith('filtered_aligned/')]
    if filtered_rows.empty:
        return False
    missing = [p for p in filtered_rows['file_name'].astype(str) if not os.path.exists(os.path.join(BASE, p))]
    if missing:
        print('Filtered files missing from existing manifest:', missing[:5])
        return False
    return True


def prepare_ecc_gray(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape[:2]
    scale = min(1.0, ECC_SCORE_SIZE / max(h, w))
    if scale < 1.0:
        gray = cv2.resize(gray, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    gray = gray.astype(np.float32) / 255.0
    return gray


def ecc_warp_is_sane(warp, shape):
    h, w = shape[:2]
    corners = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]]).reshape(-1, 1, 2)
    warped = cv2.perspectiveTransform(corners, warp).reshape(-1, 2)
    area = abs(cv2.contourArea(warped.astype(np.float32)))
    src_area = max(1.0, float(w * h))
    scale = area / src_area
    if scale < ECC_MIN_SCALE or scale > ECC_MAX_SCALE:
        return False
    drift = np.linalg.norm(warped - corners.reshape(-1, 2), axis=1)
    max_allowed = ECC_MAX_CORNER_DRIFT_FRAC * max(h, w)
    if float(drift.max()) > max_allowed:
        return False
    return True


def ecc_homography_score(reference_img, moving_img):
    ref_gray = prepare_ecc_gray(reference_img)
    mov_gray = prepare_ecc_gray(moving_img)
    if ref_gray.shape != mov_gray.shape:
        mov_gray = cv2.resize(mov_gray, (ref_gray.shape[1], ref_gray.shape[0]), interpolation=cv2.INTER_AREA)
    baseline_mse = float(np.mean((ref_gray - mov_gray) ** 2))

    warp = np.eye(3, 3, dtype=np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, ECC_MAX_ITERS, ECC_EPS)
    try:
        cc, warp = cv2.findTransformECC(ref_gray, mov_gray, warp, cv2.MOTION_HOMOGRAPHY, criteria)
        if not ecc_warp_is_sane(warp, ref_gray.shape):
            return -baseline_mse
        warped = cv2.warpPerspective(
            mov_gray,
            warp,
            (ref_gray.shape[1], ref_gray.shape[0]),
            flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP,
        )
        mse = float(np.mean((ref_gray - warped) ** 2))
        if mse > baseline_mse * ECC_MAX_WORSE_FACTOR:
            return -baseline_mse
        return float(cc) - mse
    except cv2.error:
        return -baseline_mse


def choose_best_duplicate(rows):
    if len(rows) == 1:
        return rows[0], {'representative_score': 1.0}

    images = []
    usable_rows = []
    for row in rows:
        path = os.path.join(BASE, row['file_name'])
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is not None:
            usable_rows.append(row)
            images.append(img)

    if len(usable_rows) == 1:
        return usable_rows[0], {'representative_score': 1.0}
    if not usable_rows:
        return rows[0], {'representative_score': float('nan')}

    scores = []
    for i, ref_img in enumerate(images):
        pair_scores = []
        for j, mov_img in enumerate(images):
            if i == j:
                continue
            pair_scores.append(ecc_homography_score(ref_img, mov_img))
        scores.append(float(np.mean(pair_scores)) if pair_scores else 1.0)

    best_idx = int(np.nanargmax(scores))
    return usable_rows[best_idx], {'representative_score': scores[best_idx]}


def run_filtered(aligned_df):
    if REDO_FILTERED:
        print('REDO_FILTERED=True; deleting previous filtered outputs.')
        replace_dir(FILTERED_DIR)
    elif filtered_complete():
        print('Filtered dataset already complete; skipping.')
        return pd.read_csv(MANIFEST_CSV)
    else:
        os.makedirs(FILTERED_DIR, exist_ok=True)

    if aligned_df.empty:
        out_df = write_manifest(pd.DataFrame(columns=MANIFEST_COLUMNS))
        return out_df

    records = []
    aligned_candidates = aligned_df[aligned_df['status'] == 'aligned'].copy() if 'status' in aligned_df.columns else aligned_df.copy()
    non_aligned = aligned_df[aligned_df['status'] != 'aligned'].copy() if 'status' in aligned_df.columns else pd.DataFrame()
    records.extend([dict(row) for _, row in non_aligned.iterrows()])

    group_cols = ['location', 'time_of_day', 'weather']
    for key, group in tqdm(list(aligned_candidates.groupby(group_cols, sort=True)), desc='Filter duplicates'):
        location, tod, weather = key
        rows = [row for _, row in group.iterrows()]
        selected, stats = choose_best_duplicate(rows)
        selected_source = selected['source_file']

        src_path = os.path.join(BASE, selected['file_name'])
        img = cv2.imread(src_path, cv2.IMREAD_COLOR)
        if img is None:
            print(f'[MISS] filtered source missing: {src_path}')
            for row in rows:
                records.append(manifest_record_from_row(row, 'filter_failed', 'missing_aligned_file'))
            continue

        location_dir = os.path.join(FILTERED_DIR, clean_stem(location))
        os.makedirs(location_dir, exist_ok=True)
        out_name = unique_output_name(location_dir, f'{clean_stem(location)}_{clean_stem(tod)}_{clean_stem(weather)}')
        out_path = os.path.join(location_dir, out_name)
        cv2.imwrite(out_path, img, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        filtered_file = os.path.relpath(out_path, BASE)

        for row in rows:
            if row['source_file'] == selected_source:
                rec = dict(row)
                rec['file_name'] = filtered_file
                rec['caption'] = caption_from_labels(location, tod, weather)
                rec['representative_score'] = stats['representative_score']
                rec['status'] = 'kept'
                rec['drop_reason'] = ''
            else:
                rec = dict(row)
                rec['status'] = 'duplicate_filtered'
                rec['drop_reason'] = 'duplicate_condition'
                rec['representative_score'] = stats['representative_score']
            records.append(rec)

    out_df = write_manifest(pd.DataFrame(records))
    print(f'\nFiltered stage wrote {len(out_df)} final rows to {MANIFEST_CSV}')
    return out_df


final_manifest_df = run_filtered(aligned_df)
display(final_manifest_df.head())
summarize_manifest(final_manifest_df, 'Final manifest summary', path_base=BASE)
kept_df = final_manifest_df[final_manifest_df['status'] == 'kept'].copy()
show_image_sample(kept_df, BASE, title='Kept final image sample')


## Hugging Face Export And Preview

Export the final filtered dataset to an ImageFolder-style directory with `metadata.jsonl` captions, then show a quick visual contact sheet from one filtered location.

In [ ]:
def hf_dataset_complete():
    metadata_path = os.path.join(HF_DATASET_DIR, 'metadata.jsonl')
    images_dir = os.path.join(HF_DATASET_DIR, 'images')
    if not file_exists_and_nonempty(metadata_path) or not os.path.isdir(images_dir):
        return False
    try:
        meta = pd.read_json(metadata_path, lines=True)
    except Exception:
        return False
    if meta.empty or not {'file_name', 'text'}.issubset(meta.columns):
        return False
    missing = [p for p in meta['file_name'].astype(str) if not os.path.exists(os.path.join(HF_DATASET_DIR, p))]
    if missing:
        print('HF dataset files missing:', missing[:5])
        return False
    return True

def run_hf_export(final_manifest_df):
    if REDO_HF_DATASET:
        print('REDO_HF_DATASET=True; deleting previous HF dataset export.')
        replace_dir(HF_DATASET_DIR)
    elif hf_dataset_complete():
        print('HF dataset already complete; skipping.')
        return
    else:
        os.makedirs(HF_DATASET_DIR, exist_ok=True)

    images_dir = os.path.join(HF_DATASET_DIR, 'images')
    os.makedirs(images_dir, exist_ok=True)
    metadata_rows = []

    kept_rows = final_manifest_df[final_manifest_df['status'] == 'kept'].copy() if 'status' in final_manifest_df.columns else final_manifest_df.copy()
    for _, row in kept_rows.iterrows():
        src_path = os.path.join(BASE, row['file_name'])
        if not os.path.exists(src_path):
            print(f"[MISS] HF source missing: {src_path}")
            continue
        out_name = unique_output_name(images_dir, clean_stem(row['file_name']))
        dst_rel = os.path.join('images', out_name)
        dst_path = os.path.join(HF_DATASET_DIR, dst_rel)
        shutil.copy2(src_path, dst_path)
        metadata_rows.append({
            'file_name': dst_rel,
            'text': row['caption'],
            'location': row['location'],
            'time_of_day': row['time_of_day'],
            'weather': row['weather'],
            'is_synthetic': row.get('is_synthetic', ''),
            'split': row['split'],
        })

    metadata_path = os.path.join(HF_DATASET_DIR, 'metadata.jsonl')
    with open(metadata_path, 'w') as f:
        for rec in metadata_rows:
            f.write(json.dumps(rec) + '\n')
    pd.DataFrame(metadata_rows).to_csv(os.path.join(HF_DATASET_DIR, 'metadata.csv'), index=False)
    print(f'Exported {len(metadata_rows)} HF dataset rows to {HF_DATASET_DIR}')

run_hf_export(final_manifest_df)

print('\nFinal dataset summary')
print('---------------------')
raw_count = sum(1 for f in os.listdir(RAW_DIR) if os.path.splitext(f)[1].lower() in VALID_IMAGE_SUFFIXES and ':zone.identifier' not in f.lower()) if os.path.isdir(RAW_DIR) else 0
processed_count = len(manifest_df)
aligned_count = len(aligned_df)
filtered_count = int((final_manifest_df['status'] == 'kept').sum()) if 'status' in final_manifest_df.columns else len(final_manifest_df)
print('raw images:', raw_count)
print('processed images:', processed_count)
print('aligned images:', aligned_count)
print('filtered final images:', filtered_count)
print('failed alignment count:', len(alignment_failures))
print('dropped-for-crop-loss count:', len(crop_drops))
print('hf dataset images:', len(list(Path(os.path.join(HF_DATASET_DIR, 'images')).glob('*.jpg'))) if os.path.isdir(os.path.join(HF_DATASET_DIR, 'images')) else 0)
